In [ ]:
from sls_client import get_sls_data_by_query

query="""
en:wbv and ap:/merchant/store/account/authorization/query-status | select case when status=0 then '已预录入但未授权' when status=1 then '取消授权' when status=2 then '已授权' else '未授权' end as "授权状态"
,date_of_year as "日期",uid,phone from (select cast(json_extract(json_extract_scalar(ai, '$.rt'),'$.data.status') as bigint) status, 
json_extract(json_extract_scalar(ai, '$.rt'),'$.data.displayed') displayed,uid,phone,date_format(__time__, '%Y%m%d') date_of_year from log limit 1000000) group by 1,2,3,4 limit 1000000
"""

from datetime import datetime,timedelta
yesterday = (datetime.now() - timedelta(days=1))
# 将yesterday的时分秒设置为0点0分0秒
yesterday = yesterday.replace(hour=0, minute=0, second=0, microsecond=0)
today=datetime.now()
# 将today的时分秒设置为0点0分0秒
today = today.replace(hour=0, minute=0, second=0, microsecond=0)

users_df=get_sls_data_by_query(project="xianmu-front-end-log",logstore="xm-mall", query=query, from_time=yesterday, to_time=today)

In [ ]:
pay_channel_query = """
ap:"/payment/union/pay" and en:mp| select case json_extract_scalar(ai, '$.rt.data.channelType') 
    when 'FIRE_FACE_WECHAT' then 'B2B支付' 
    when 'WECHAT_NATIVE' then '微信直连' 
    when 'DIN_WECHAT_MP' then '智付' 
    when 'BILL' then '账期' else '鲜沐卡' end as "支付渠道",uid,phone 
    from log where json_extract(ai, '$.rt.data') is not null 
    group by 1,2,3 limit 1000000
"""

pay_channel_df = get_sls_data_by_query(
    project="xianmu-front-end-log",
    logstore="xm-mall",
    query=pay_channel_query,
    from_time=yesterday,
    to_time=today,
)

In [ ]:
# 找出来哪些授权了，但是没有使用B2B支付的用户：


merged_df=pay_channel_df.merge(users_df, on="phone", how="inner")

merged_df[(merged_df['授权状态']=='已授权') & (merged_df['支付渠道']!='B2B支付')]

In [8]:
import pandas as pd

accounts=pd.read_csv("/Users/pengtang/Desktop/has_orders_last30days_account_id.csv")

In [ ]:
import requests

headers = {"token": "mall__eb3ee8f1-5668-4097-9656-d6b4e2a32c99"}

failed_accounts = []

import threading
import queue
from concurrent.futures import ThreadPoolExecutor

# 创建一个队列来存储结果
result_queue = queue.Queue()


def process_account(account_id, headers):
    """处理单个account的预录入"""
    url = f"https://h5.summerfarm.net/merchant/store/account/authorization/pre-entry?accountId={account_id}"
    try:
        result = requests.post(url=url, headers=headers).json()
        if not result["data"]:
            print(f"account预录入失败了: {account_id}, result: {result}")
            result_queue.put(account_id)  # 将失败的account_id放入队列
        else:
            print(f"account预录入成功了: {account_id}")
    except Exception as e:
        print(f"account预录入出错: {account_id}, error: {e}")
        result_queue.put(account_id)


# 使用ThreadPoolExecutor管理线程
with ThreadPoolExecutor(max_workers=3) as executor:
    for _index, row in accounts.iloc[
        2000:
    ].iterrows():  # 从第2001行开始，使用iloc[2000:]
        account_id = row["account_id"]
        executor.submit(process_account, account_id, headers)

# 从队列中获取所有失败的account_id
failed_accounts = []
while not result_queue.empty():
    failed_accounts.append(result_queue.get())

print(f"失败了{len(failed_accounts)}个account，失败的列表:{failed_accounts}")

In [1]:
import pandas as pd

user_un_autherised_df = pd.read_csv("/Users/pengtang/Desktop/未授权的客户分析.csv")

user_un_autherised_df.describe()

,m_id,account_id,三十天订单数,三十天下单GMV
count,35702.000000,35702.000000,35702.000000,27576.000000
mean,333677.606129,376413.346339,2.849644,1760.696073
std,156092.585625,169356.440850,4.430229,4224.798694
min,13.000000,13.000000,0.000000,0.000000
25%,215188.750000,251178.250000,1.000000,298.000000
50%,374239.000000,426600.000000,1.000000,717.000000
75%,471941.500000,525534.000000,3.000000,1746.250000
max,523498.000000,574982.000000,163.000000,186028.000000


In [2]:
user_un_autherised_df.head(1)

,m_id,account_id,店铺名称,门店城市,是否单店,三十天订单数,最后下单日,三十天下单GMV
0,56316,68246,Hibake春熙路店,成都市,大客户,20,2025-03-21,186028.0


In [9]:
city_df = (
    user_un_autherised_df.groupby(["门店城市"])
    .aggregate({"m_id": "count", "三十天订单数": "sum", "三十天下单GMV": "sum"})
    .reset_index()
)

city_df = city_df.sort_values(by=["三十天下单GMV"], ascending=False)

In [10]:
total_gmv=user_un_autherised_df['三十天下单GMV'].sum()

city_df['GMV占比']=city_df['三十天下单GMV']/total_gmv
city_df

,门店城市,m_id,三十天订单数,三十天下单GMV,GMV占比
0,上海市,2090,7848,3611189.48,0.074376
61,杭州市,1670,6123,3008441.08,0.061962
106,苏州市,1661,6181,2618554.46,0.053932
81,深圳市,1332,4238,2272605.30,0.046807
122,重庆市,1256,4002,1865041.65,0.038413
...,...,...,...,...,...
93,玉溪市,2,0,0.00,0.000000
130,韶关市,1,0,0.00,0.000000
46,恩施土家族苗族自治州,1,0,0.00,0.000000
66,楚雄彝族自治州,1,0,0.00,0.000000


In [20]:
for gmv in range(1000, 20000, 1000):
    _df = user_un_autherised_df[user_un_autherised_df["三十天下单GMV"] > gmv]
    print(
        f"最近30天GMV大于:{gmv}的客户数:{_df.shape[0]}, GMV占比:{round(100.0*_df['三十天下单GMV'].sum()/total_gmv,2)}%, 近30天GMV:{round(_df['三十天下单GMV'].sum()/10000)}万"
    )

最近30天GMV大于:1000的客户数:11069, GMV占比:85.99%, 近30天GMV:4175万
最近30天GMV大于:2000的客户数:5994, GMV占比:71.18%, 近30天GMV:3456万
最近30天GMV大于:3000的客户数:3775, GMV占比:59.99%, 近30天GMV:2913万
最近30天GMV大于:4000的客户数:2654, GMV占比:52.01%, 近30天GMV:2525万
最近30天GMV大于:5000的客户数:1947, GMV占比:45.49%, 近30天GMV:2209万
最近30天GMV大于:6000的客户数:1479, GMV占比:40.2%, 近30天GMV:1952万
最近30天GMV大于:7000的客户数:1166, GMV占比:36.03%, 近30天GMV:1749万
最近30天GMV大于:8000的客户数:956, GMV占比:32.79%, 近30天GMV:1592万
最近30天GMV大于:9000的客户数:776, GMV占比:29.64%, 近30天GMV:1439万
最近30天GMV大于:10000的客户数:660, GMV占比:27.38%, 近30天GMV:1329万
最近30天GMV大于:11000的客户数:572, GMV占比:25.49%, 近30天GMV:1238万
最近30天GMV大于:12000的客户数:477, GMV占比:23.24%, 近30天GMV:1129万
最近30天GMV大于:13000的客户数:419, GMV占比:21.75%, 近30天GMV:1056万
最近30天GMV大于:14000的客户数:368, GMV占比:20.33%, 近30天GMV:987万
最近30天GMV大于:15000的客户数:323, GMV占比:18.98%, 近30天GMV:922万
最近30天GMV大于:16000的客户数:277, GMV占比:17.51%, 近30天GMV:850万
最近30天GMV大于:17000的客户数:244, GMV占比:16.38%, 近30天GMV:796万
最近30天GMV大于:18000的客户数:221, GMV占比:15.55%, 近30天GMV:755万
最近30天GMV大于:19000的客户数:198, GMV占比:14.

In [22]:
city_over_2000_df = (
    user_un_autherised_df[user_un_autherised_df["三十天下单GMV"] > 3000]
    .groupby(["门店城市"])
    .aggregate({"m_id": "count", "三十天订单数": "sum", "三十天下单GMV": "sum"})
    .reset_index()
)

city_over_2000_df = city_over_2000_df.sort_values(by=["三十天下单GMV"], ascending=False)
city_over_2000_df.columns = ["门店城市", "门店数", "三十天订单总数", "三十天总下单GMV"]
city_over_2000_df

,门店城市,门店数,三十天订单总数,三十天总下单GMV
0,上海市,275,3772,2493963.57
51,杭州市,233,2974,2052744.56
87,苏州市,237,2921,1666381.96
65,深圳市,195,1965,1514949.31
101,重庆市,158,1615,1156270.42
...,...,...,...,...
49,普洱市,1,7,5337.00
98,邵阳市,1,7,5217.08
56,永州市,1,8,4068.49
100,鄂州市,1,11,3497.00


In [4]:
import json


# 从文件加载 JSON 数据: /Users/pengtang/major_price.json
with open("/Users/pengtang/major_price.json", "r") as f:
    data = json.load(f)

print(len(data['data']['list'][0]))

7


In [7]:
import pandas as pd

major_price_df=pd.DataFrame(data['data']['list'][0])
major_price_df.describe()

,largeAreaNo
count,1924.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


In [8]:
major_price_df.head(1)

,largeAreaName,largeAreaNo,majorPrices,pdName,sku,specification,specificationUnit
0,杭州大区,1,"{'areaName': '杭州', 'areaNo': 1001, 'categoryId...",广东粗皮香水柠檬,16788463466,净重5-5.1斤/一级/（大果）90-150克/-(受季节影响，春果部分会存在皮厚汁水少，属...,包
